# Compare Splitting Results Across Parameter Types - NonLinLoc Catalog, Station AXAS2, in the week around the 2015 Axial Seamount Eruption

In [ ]:
# Dependencies

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy.core.utcdatetime import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import read_events
import os
import sys
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add local swspy directory to path (before other imports)
swspy_local_path = os.path.abspath('../swspy')
if swspy_local_path not in sys.path:
    sys.path.insert(0, swspy_local_path)

# Import swspy from local directory
import swspy

# Add scripts directory to path for custom modules
sys.path.append('.')
from get_all_traces import get_station_traces_batch
from splitting_functions import *
from teanby_clustering import *

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")
print(f"ObsPy version: {obspy.__version__}")
print(f"SWSPy available: {'Yes' if 'swspy' in sys.modules else 'No'}")
print(f"SWSPy location: {swspy.__file__}")

In [ ]:
# Updated plotting functions that work with our dataframe structure

def plot_fast_direction_rose(results_df, title="Fast Direction Distribution", 
                              nbins=18, figsize=(8, 8), color='steelblue',
                              edgecolor='black', linewidth=0.5):
    """
    Create a polar rose plot (histogram) of fast directions from splitting results.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results containing 'phi' column
    title : str
        Title for the plot
    nbins : int
        Number of angular bins (default 18 = 10° bins for ±90°)
    figsize : tuple
        Figure size (width, height)
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    """
    # Extract fast directions (phi) from results
    fast_directions = np.deg2rad(results_df['phi'].values)
    
    # Create polar histogram
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='polar')
    
    # Create histogram bins (-pi/2 to pi/2 for -90° to +90°)
    bins = np.linspace(-np.pi/2, np.pi/2, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(-1)
    
    # Set angular limits (-90° to +90°)
    ax.set_thetamin(-90)
    ax.set_thetamax(90)
    
    # Set radial ticks
    ax.set_rlabel_position(0)
    
    # Add degree labels
    tick_labels = ['-90°', '-60°', '-30°', '0°', '30°', '60°', '90°']
    tick_positions = np.deg2rad([-90, -60, -30, 0, 30, 60, 90])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    ax.set_ylim(0, 375)
    
    # Add title with statistics
    n_measurements = len(fast_directions)
    # Calculate circular mean for ±90° range
    mean_direction = np.rad2deg(np.arctan2(np.sin(fast_directions).sum(), 
                                           np.cos(fast_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    return fig, ax


def plot_splitting_timeseries_smooth(results_df, station='AXAS2', 
                                     figsize=(14, 8), x_width_days=5, x_overlap=0.95,
                                     y_width_phi=5, y_width_dt=2, y_overlap=0.95,
                                     sigma=2.0, sampling_rate=200.0,
                                     time_column='event_datetime'):
    """
    Create smoothed 2D histogram time-series plots inspired by Baillard's approach.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results containing 'phi', 'dt', and time columns
    station : str
        Station name for title
    figsize : tuple
        Figure size (width, height)
    x_width_days : float
        Width of moving time window in days
    x_overlap : float
        Overlap fraction for time windows (0.95 = 95% overlap)
    y_width_phi : float
        Bin width for phi in degrees
    y_width_dt : int
        Bin width for dt in samples (1 sample = 1/sampling_rate seconds)
    y_overlap : float
        Overlap for smoothing in y-direction
    sigma : float
        Gaussian smoothing parameter
    sampling_rate : float
        Sampling rate in Hz (default 200 Hz)
    time_column : str
        Name of the time column in results_df (default 'origin_time')
    """
    import matplotlib.gridspec as gridspec
    from matplotlib.dates import DateFormatter
    import matplotlib.dates as mdates
    from scipy.ndimage import gaussian_filter
    
    # Prepare data from DataFrame
    df = results_df.copy()
    df['time'] = pd.to_datetime(df[time_column])
    df['phi_deg'] = df['phi']
    df['phi_rad'] = np.deg2rad(df['phi'])
    df['phi_rad_norm'] = ((df['phi_rad'] + np.pi/2) % np.pi) - np.pi/2
    df['dt_seconds'] = df['dt']
    df['dt_samples'] = df['dt'] * sampling_rate
    df = df.sort_values('time')
    
    if len(df) == 0:
        print("No data to plot")
        return
    
    # Create figure with gridspec layout
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2, 1, hspace=0.3, top=0.9, bottom=0.1, 
                          left=0.08, right=0.95)
    
    ax_phi = plt.subplot(gs[0])
    ax_dt = plt.subplot(gs[1], sharex=ax_phi)
    
    # Convert datetime to matplotlib date numbers
    time_nums = mdates.date2num(df['time'])
    
    # === PHI PLOT (in radians) ===
    # Create bins
    time_range = (time_nums.min(), time_nums.max())
    n_time_bins = int((time_range[1] - time_range[0]) / (x_width_days * (1 - x_overlap)))
    n_time_bins = max(20, min(n_time_bins, 100))  # Reasonable limits
    
    # Phi bins from -pi/2 to +pi/2 radians (-1.57 to +1.57)
    phi_bins = np.arange(-np.pi/2, np.pi/2 + y_width_phi*np.pi/180, y_width_phi*np.pi/180)
    n_phi_bins = len(phi_bins) - 1
    
    # Create 2D histogram for phi
    H_phi, xedges_phi, yedges_phi = np.histogram2d(
        time_nums, df['phi_rad_norm'], 
        bins=[n_time_bins, phi_bins],
        range=[time_range, None]
    )
    
    # Normalize by column (each time bin) - "norm_y=True" in Baillard's code
    H_phi_norm = H_phi.copy()
    for i in range(H_phi.shape[0]):
        col_sum = H_phi[i, :].sum()
        if col_sum > 0:
            H_phi_norm[i, :] = H_phi[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_phi_smooth = gaussian_filter(H_phi_norm, sigma=sigma)
    
    # Mask zeros
    H_phi_smooth = np.ma.masked_where(H_phi_smooth < 0.001, H_phi_smooth)
    
    # Plot with imshow for smooth appearance
    extent_phi = [xedges_phi[0], xedges_phi[-1], yedges_phi[0], yedges_phi[-1]]
    im_phi = ax_phi.imshow(H_phi_smooth.T, 
                           origin='lower',
                           aspect='auto',
                           extent=extent_phi,
                           cmap='magma',
                           interpolation='bilinear',
                           alpha=0.9)
    
    # Colorbar
    cbar_phi = plt.colorbar(im_phi, ax=ax_phi, pad=0.01)
    cbar_phi.set_label('Normalized Density', fontsize=10)
    
    # Format phi axis with degrees
    ax_phi.set_ylabel('Fast Direction φ (°)', fontsize=12, fontweight='bold')
    ax_phi.set_ylim(-np.pi/2, np.pi/2)
    ax_phi.axhline(0, color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    ax_phi.grid(True, alpha=0.3, linestyle='--', color='white')
    
    # Add axvline dashed white line at time = 2015-04-24 05:00:00
    event_time_line = pd.to_datetime('2015-04-24 05:00:00')
    ax_phi.axvline(mdates.date2num(event_time_line), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Set y-ticks in radians but label with degrees
    phi_ticks_rad = np.array([-np.pi/2, -np.pi/3, -np.pi/6, 0, np.pi/6, np.pi/3, np.pi/2])
    phi_ticks_deg = np.array([-90, -60, -30, 0, 30, 60, 90])
    ax_phi.set_yticks(phi_ticks_rad)
    ax_phi.set_yticklabels([f'{int(d)}°' for d in phi_ticks_deg])
    
    # Add moving average
    window_size = max(5, len(df) // 10)
    if len(df) >= window_size:
        df['phi_rad_ma'] = df['phi_rad_norm'].rolling(window=window_size, center=True).mean()
        ax_phi.plot(df['time'], df['phi_rad_ma'], 'w-', linewidth=2.5, alpha=0.9)
        ax_phi.plot(df['time'], df['phi_rad_ma'], 'black', linewidth=2,
                   label=f'{window_size}-event moving avg', alpha=0.8)
        ax_phi.legend(loc='upper right', fontsize=9, facecolor='white', 
                     edgecolor='white', framealpha=0.7)
    
    # === DT PLOT (in samples) ===
    dt_max_samples = df['dt_samples'].quantile(0.98)
    # Create bins in samples
    dt_bins = np.arange(0, min(40, dt_max_samples) + y_width_dt, y_width_dt)
    n_dt_bins = len(dt_bins) - 1
    
    # Create 2D histogram for dt
    H_dt, xedges_dt, yedges_dt = np.histogram2d(
        time_nums, df['dt_samples'],
        bins=[n_time_bins, dt_bins],
        range=[time_range, None]
    )
    
    # Normalize by column
    H_dt_norm = H_dt.copy()
    for i in range(H_dt.shape[0]):
        col_sum = H_dt[i, :].sum()
        if col_sum > 0:
            H_dt_norm[i, :] = H_dt[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_dt_smooth = gaussian_filter(H_dt_norm, sigma=sigma)
    
    # Mask zeros
    H_dt_smooth = np.ma.masked_where(H_dt_smooth < 0.001, H_dt_smooth)
    
    # Plot with imshow
    extent_dt = [xedges_dt[0], xedges_dt[-1], yedges_dt[0], yedges_dt[-1]]
    im_dt = ax_dt.imshow(H_dt_smooth.T,
                         origin='lower',
                         aspect='auto',
                         extent=extent_dt,
                         cmap='magma',
                         interpolation='bilinear',
                         alpha=0.9)
    
    # Colorbar
    cbar_dt = plt.colorbar(im_dt, ax=ax_dt, pad=0.01)
    cbar_dt.set_label('Normalized Density', fontsize=10)
    
    # Add axvline dashed white line at time = 2015-04-24 05:00:00
    event_time_line = pd.to_datetime('2015-04-24 05:00:00')
    ax_dt.axvline(mdates.date2num(event_time_line), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Format dt axis with samples
    ax_dt.set_ylabel('Delay Time δt (samples)', fontsize=12, fontweight='bold')
    ax_dt.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax_dt.set_ylim(0, dt_max_samples)  # Show up to 25 samples (0.125 sec at 200 Hz)
    ax_dt.grid(True, alpha=0.3, linestyle='--', color='white')
    
    # Add moving average for dt
    if len(df) >= window_size:
        df['dt_samples_ma'] = df['dt_samples'].rolling(window=window_size, center=True).mean()
        ax_dt.plot(df['time'], df['dt_samples_ma'], 'w-', linewidth=2.5, alpha=0.9)
        ax_dt.plot(df['time'], df['dt_samples_ma'], 'black', linewidth=2,
                  label=f'{window_size}-event moving avg', alpha=0.8)
        ax_dt.legend(loc='upper right', fontsize=9, facecolor='white',
                    edgecolor='white', framealpha=0.7)
    
    # Format x-axis with dates
    date_formatter = DateFormatter('%Y-%m-%d')
    ax_dt.xaxis.set_major_formatter(date_formatter)
    
    # Auto-adjust date locator
    days_span = (df['time'].max() - df['time'].min()).days
    if days_span > 60:
        ax_dt.xaxis.set_major_locator(mdates.MonthLocator())
    elif days_span > 14:
        ax_dt.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    else:
        ax_dt.xaxis.set_major_locator(mdates.DayLocator())
    
    plt.setp(ax_dt.xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.setp(ax_phi.xaxis.get_majorticklabels(), visible=False)
    
    # Add title with statistics (in both degrees and radians)
    phi_mean_deg = df['phi_deg'].mean()
    phi_std_deg = df['phi_deg'].std()
    dt_mean_sec = df['dt_seconds'].mean()
    dt_std_sec = df['dt_seconds'].std()
    
    title = (f"Splitting Parameter Time Series - Station {station}\n"
             f"N = {len(df)} events | "
             f"φ: {phi_mean_deg:.1f}° ± {phi_std_deg:.1f}° "
             f"({np.deg2rad(phi_mean_deg):.2f} ± {np.deg2rad(phi_std_deg):.2f} rad) | "
             f"δt: {dt_mean_sec:.3f} ± {dt_std_sec:.3f} s "
             f"({dt_mean_sec*sampling_rate:.1f} ± {dt_std_sec*sampling_rate:.1f} samples)")
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    return fig, (ax_phi, ax_dt), df


def plot_fast_direction_rose_eruption_comparison(results_df, eruption_time=None, 
                                                  title_prefix="Fast Direction Distribution",
                                                  nbins=36, figsize=(16, 7), color='steelblue',
                                                  edgecolor='black', linewidth=0.5,
                                                  time_column='event_datetime'):
    """
    Create side-by-side 360° rose plots comparing fast directions before and after eruption.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results containing 'phi' and time columns
    eruption_time : UTCDateTime or str
        Time of eruption onset (default: 2015-04-24T06:00:00)
    title_prefix : str
        Prefix for plot titles
    nbins : int
        Number of angular bins (default 36 = 10° bins for 360°)
    figsize : tuple
        Figure size (width, height) for combined plot
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    time_column : str
        Name of the time column in results_df (default 'event_datetime')
    """
    if eruption_time is None:
        eruption_time = UTCDateTime(2015, 4, 24, 6)
    elif not isinstance(eruption_time, UTCDateTime):
        eruption_time = UTCDateTime(eruption_time)
    
    # Convert time column to UTCDateTime for comparison
    df_time = results_df[time_column].apply(lambda x: UTCDateTime(x) if not isinstance(x, UTCDateTime) else x)
    results_before = results_df[df_time < eruption_time]
    results_after = results_df[df_time >= eruption_time]
    
    # Create figure with two subplots
    fig = plt.figure(figsize=figsize)
    
    # Before eruption plot (left)
    ax1 = fig.add_subplot(121, projection='polar')
    plot_rose_subplot(results_before, ax1, f"{title_prefix}\nBefore Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    # After eruption plot (right)
    ax2 = fig.add_subplot(122, projection='polar')
    plot_rose_subplot(results_after, ax2, f"{title_prefix}\nAfter Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    plt.tight_layout()
    return fig, (ax1, ax2), (results_before, results_after)


def plot_rose_subplot(results_df, ax, title, nbins, color, edgecolor, linewidth):
    """
    Helper function to plot rose diagram on a given axis.
    """
    if len(results_df) == 0:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, 
                ha='center', va='center', fontsize=14)
        return
    
    fast_directions = []
    for phi in results_df['phi'].values:  # in degrees (-90 to +90)
        # Convert to 0-360 range and add 180° symmetry
        phi_0_180 = phi + 90  # Convert to 0-180 range
        phi_rad_1 = np.deg2rad(phi_0_180)
        phi_rad_2 = np.deg2rad(phi_0_180 + 180)  # Add 180° symmetric value
        fast_directions.append(phi_rad_1)
        fast_directions.append(phi_rad_2)
    
    fast_directions = np.array(fast_directions)
    
    # Create histogram bins (0 to 2π for 0° to 360°)
    bins = np.linspace(0, 2*np.pi, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = 2 * np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(-1)
    
    # Set radial ticks
    ax.set_rlabel_position(45)
    
    # Add degree labels for full 360°
    tick_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
    tick_positions = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Adjust y-limit
    max_count = counts.max()
    ax.set_ylim(0, max_count * 1.2)
    
    # Add title with statistics
    n_measurements = len(fast_directions) // 2  # Divide by 2 since we doubled for symmetry
    # Calculate circular mean
    original_directions = np.deg2rad(results_df['phi'].values)
    mean_direction = np.rad2deg(np.arctan2(np.sin(original_directions).sum(), 
                                           np.cos(original_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)


def plot_fast_direction_rose_360(results_df, title="Fast Direction Distribution", 
                                  nbins=36, figsize=(8, 8), color='steelblue',
                                  edgecolor='black', linewidth=0.5):
    """
    Create a 360° polar rose plot (histogram) of fast directions with 180° symmetry.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results containing 'phi' column
    title : str
        Title for the plot
    nbins : int
        Number of angular bins (default 36 = 10° bins for 360°)
    figsize : tuple
        Figure size (width, height)
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    """
    # Extract fast directions (phi) from results
    fast_directions = []
    for phi in results_df['phi'].values:  # in degrees (-90 to +90)
        # Convert to 0-360 range and add 180° symmetry
        phi_0_180 = phi + 90  # Convert to 0-180 range
        phi_rad_1 = np.deg2rad(phi_0_180)
        phi_rad_2 = np.deg2rad(phi_0_180 + 180)  # Add 180° symmetric value
        fast_directions.append(phi_rad_1)
        fast_directions.append(phi_rad_2)
    
    fast_directions = np.array(fast_directions)
    
    # Create polar histogram
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='polar')
    
    # Create histogram bins (0 to 2π for 0° to 360°)
    bins = np.linspace(0, 2*np.pi, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = 2 * np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from East)
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(-1)
    
    # Set radial ticks
    ax.set_rlabel_position(45)
    
    # Add degree labels for full 360°
    tick_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
    tick_positions = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Adjust y-limit (counts are doubled due to symmetry)
    max_count = counts.max()
    ax.set_ylim(0, max_count * 1.2)
    
    # Add title with statistics
    n_measurements = len(fast_directions) // 2  # Divide by 2 since we doubled for symmetry
    # Calculate circular mean for original ±90° range
    original_directions = np.deg2rad(results_df['phi'].values)
    mean_direction = np.rad2deg(np.arctan2(np.sin(original_directions).sum(), 
                                           np.cos(original_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    return fig, ax

In [ ]:
# Load results
results_df = pd.read_csv('../results/splitting_results_swspy_axas2_april_20_28_2_sigma_start.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df,
    title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df,
    title_prefix=f"Fast Direction Distribution, AXAS2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df,
    figsize=(12,8),
    station='AXAS2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2,
)
plt.show()

In [ ]:
# Load results
results_df_val = pd.read_csv('../results/splitting_results_axas2_apr_20_28_sigma_2_validation.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df_val,
    title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End; Validation",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_val,
    title_prefix=f"Fast Direction Distribution, AXAS2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End; Validation",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df_val,
    figsize=(12,8),
    station='AXAS2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End; Validation',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2,
)
plt.show()

In [ ]:
# Load results
results_df_val = pd.read_csv('../results/splitting_results_axas2_apr_20_28_sigma_2_validation_orig_st_trim.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df_val,
    title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End; Validation; Original Stream Trimmed",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_val,
    title_prefix=f"Fast Direction Distribution, AXAS2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End; Validation; Original Stream Trimmed",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df_val,
    figsize=(12,8),
    station='AXAS2, SWSPy: 2σ to S-pick Start, 1.5-2.5 Tmid End; Validation; Original Stream Trimmed',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2,
)
plt.show()

In [ ]:
# Load results
results_df_1_tmid_start = pd.read_csv('../results/splitting_results_axas2_apr_20_28_1_tmid_start.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df_1_tmid_start,
    title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 2σ to S-pick Start, 1.0-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_1_tmid_start,
    title_prefix=f"Fast Direction Distribution, AXAS2, SWSPy: 2σ to S-pick Start, 1.0-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df_1_tmid_start,
    figsize=(12,8),
    station='AXAS2, SWSPy: 2σ to S-pick Start, 1.0-2.5 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2,
)
plt.show()

In [ ]:
# Load results
results_df_1_9_tmid_start_2_1_end = pd.read_csv('../results/splitting_results_axas2_apr_20_28_1_9_tmid_start_2_1_end.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df_1_9_tmid_start_2_1_end,
    title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 2σ to S-pick Start, 1.9-2.1 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_1_9_tmid_start_2_1_end,
    title_prefix=f"Fast Direction Distribution, AXAS2, SWSPy: 2σ to S-pick Start, 1.9-2.1 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df_1_9_tmid_start_2_1_end,
    figsize=(12,8),
    station='AXAS2, SWSPy: 2σ to S-pick Start, 1.9-2.1 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2,
)
plt.show()

In [ ]:
# Load results
results_df_1_8_tmid_start_2_2_end = pd.read_csv('../results/splitting_results_axas2_apr_20_28_1_8_tmid_start_2_2_end.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df_1_8_tmid_start_2_2_end,
    title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 2σ to S-pick Start, 1.8-2.2 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_1_8_tmid_start_2_2_end,
    title_prefix=f"Fast Direction Distribution, AXAS2, SWSPy: 2σ to S-pick Start, 1.8-2.2 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df_1_8_tmid_start_2_2_end,
    figsize=(12,8),
    station='AXAS2, SWSPy: 2σ to S-pick Start, 1.8-2.2 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2,
)
plt.show()

In [ ]:
# Load results
results_df_sigma_1_5_0_5_tmid_1_8_2_2 = pd.read_csv('../results/splitting_results_axas2_apr_20_28_sigma_1_5_0_5_tmid_1_8_2_2.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df_sigma_1_5_0_5_tmid_1_8_2_2,
    title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 1.5-0.5σ, 1.8-2.2 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_sigma_1_5_0_5_tmid_1_8_2_2,
    title_prefix=f"Fast Direction Distribution, AXAS2, SWSPy: 1.5-0.5σ, 1.8-2.2 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df_sigma_1_5_0_5_tmid_1_8_2_2,
    figsize=(12,8),
    station='AXAS2, SWSPy: 1.5-0.5σ, 1.8-2.2 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2,
)
plt.show()

In [ ]:


# Load results
results_df_sigma_2_1_tmid_1_8_2_2 = pd.read_csv('../results/splitting_results_axas2_apr_20_28_sigma_2_1_tmid_1_8_2_2.csv')

# Plot rose diagram
fig, ax = plot_fast_direction_rose_360(
    results_df_sigma_2_1_tmid_1_8_2_2,
    title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy: 2-1σ, 1.8-2.2 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

# Create the rose plot
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_sigma_2_1_tmid_1_8_2_2,
    title_prefix=f"Fast Direction Distribution, AXAS2, SWSPy: 2-1σ, 1.8-2.2 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

# Plot timeseries of splitting parameters with smoothing
fig, axes, df = plot_splitting_timeseries_smooth(
    results_df_sigma_2_1_tmid_1_8_2_2,
    figsize=(12,8),
    station='AXAS2, SWSPy: 2-1σ, 1.8-2.2 Tmid End',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2,
)
plt.show()

In [ ]:
import sws_methods as swm

In [ ]:
def plot_movehisto2d(x,y,
                x_label='X',y_label='Y',title='',ax=None,vmax=None,
                **movehisto2d_kwargs):
    """
    Function made to plot an histo2d but using a moving window in both directions, this ensure better
    consistency between neighbor bins. 
    The histogram can also work when data is an obspy.UTCDateTime array, then the x_start and x_end must
    be given in UTCDateTime as well and the x_width should be given in seconds.
    UTCDatetime are converted to timestamps (seconds since 1970)
    
    Inputs
    ------
        x,y: np.array: arrays containing the data to apply histogram on (x can be UTCDateTime)
        x_width,y_width: float: width of the bins (in seconds for UTCDateTime)
        [x,y]_[start,end]: float: start and end for histogram edges
        [x,y]_over: float in [0,1]: overlap for windows [1 = full overlap]
        flag_y_norm: Boolean: True to normalize by the maximum in each column
        flag_resample: Boolean: Enable resampling to uniform grid (required for smoothing)
        flag_filter: Boolean: Enable Gaussian filtering (requires flag_resample=True)
        x_filter_per, y_filter_per: float in [0,100]: width percentage for smoothing
        filter_mode: str or list: 'nearest' or 'wrap' for edge handling
        [x,y]_label: str
        
    Ouputs
    ------
        ax, im, X, Y, Z, x_bins, x_diffs
        
    Comments:
    ---------
        Uses Baillard's movehisto2d_bin for resampling and filtering
        
    UsedIn
    ------
        SWSCat.plot_movehisto2d_time, wrapper functions
    """
    
    ### Check if is made of UTCDateTimes
    
    time_flag=False
    if isinstance(x[0],UTCDateTime):
        time_flag=True
        print('X is in UTCDateTime')
        if movehisto2d_kwargs.get('x_mode','window')=='window':
            print('Remember width should be given in seconds, otherwise memory error')
    
    ### Modify x_start and x_end, and x if x is time and convert to timestamps
    x_start=movehisto2d_kwargs.get('x_start',None)
    x_end=movehisto2d_kwargs.get('x_end',None)
    y_start=movehisto2d_kwargs.get('y_start',None)
    y_end=movehisto2d_kwargs.get('y_end',None)
    
    if time_flag:
        if (x_start is not None) & (not isinstance(x_start,UTCDateTime)):
            raise ValueError('x_start must be given in obspy.UTCDateTime')
        if (x_end is not None) & (not isinstance(x_end,UTCDateTime)):
            raise ValueError('x_end must be given in obspy.UTCDateTime')
        x=np.array([value.timestamp for value in x]) # (seconds since 1970-01-01T00:00:00)
        x_start=x_start.timestamp if x_start is not None else None
        x_end=x_end.timestamp if x_end is not None else None
        movehisto2d_kwargs['x_start']=x_start
        movehisto2d_kwargs['x_end']=x_end

    ### Bin the data (smoothing handled inside movehisto2d_bin via resampling + filtering)
    
    (X,Y,Z,x_bins,x_diffs)=swm.movehisto2d_bin(x,y,**movehisto2d_kwargs)
    
    ########################
    #### Start plotting ####

    #### Grid spec
    
    bottom=0.15 if time_flag else 0.1
    
    ### Checks
    
    if ax is None:
        fig,ax = plt.subplots(gridspec_kw={'bottom':bottom,'left':0.15})
        
    if time_flag:
        X=np.array(swm.timestamp2matplotlib(X.ravel())).reshape(X.shape) # transform for plotting
        x_bins=swm.timestamp2matplotlib(x_bins)
        plt.setp( ax.xaxis.get_majorticklabels(), rotation=30 ,ha='right')
        ax.set_xlim(swm.timestamp2matplotlib([x_start,x_end]))
    else:
        ax.set_xlim([x_start,x_end])
       
    (Xm,Ym)=swm.XY2XY_pcolormesh(X,Y) # To ensure Pcolormesh will be centered on bins

    #im=ax.pcolormesh(X,Y,Z,cmap=plt.cm.get_cmap('jet'),rasterized=True)
    im=ax.pcolormesh(Xm,Ym,Z,cmap=plt.cm.get_cmap('inferno'),rasterized=True,vmax=vmax)
    
    ax.set_ylim([y_start,y_end])
    
    ax.set_aspect('auto')
    if time_flag:
        ax.xaxis_date()
            
    ### Cosmetic
    
    ax.set_ylabel(y_label) 
    if not time_flag:
        ax.set_xlabel(x_label) 
   
    ###### Return
    
    return (ax,im,X,Y,Z,x_bins,x_diffs)

In [ ]:
def plot_dt_timeseries_movehisto(results_df, time_column='event_datetime',
                                  x_width=5*24*3600, x_overlap=0.95,
                                  y_width=2, y_overlap=0.9,
                                  sampling_rate=200.0,
                                  figsize=(14, 6),
                                  station='AXAS2',
                                  flag_smooth=True,
                                  x_filter_per=10,
                                  y_filter_per=10):
    """
    Plot delay time (dt) over time using moving window 2D histogram with Baillard-style smoothing.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results
    time_column : str
        Name of time column (default 'event_datetime')
    x_width : float
        Width of time window in seconds (default 5 days = 5*24*3600)
    x_overlap : float
        Overlap fraction for time windows (0.95 = 95% overlap)
    y_width : float
        Bin width for dt in samples
    y_overlap : float
        Overlap for y-direction
    sampling_rate : float
        Sampling rate in Hz (default 200 Hz)
    figsize : tuple
        Figure size
    station : str
        Station name for title
    flag_smooth : bool
        Apply Baillard-style smoothing via resampling (default True)
    x_filter_per : float
        Gaussian filter width percentage in time direction (default 10)
    y_filter_per : float
        Gaussian filter width percentage in y direction (default 10)
    """
    import sws_methods as swm
    # Extract data
    df = results_df.copy()
    times = pd.to_datetime(df[time_column])
    dt_samples = df['dt'].values * sampling_rate
    
    # Convert to UTCDateTime for movehisto2d
    x = np.array([UTCDateTime(t) for t in times])
    y = dt_samples
    
    # Set time range
    x_start = UTCDateTime(times.min())
    x_end = UTCDateTime(times.max())
    y_start = 0
    y_end = 30

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot with Baillard-style resampling and smoothing
    (ax, im, X, Y, Z, x_bins, x_diffs) = plot_movehisto2d(
        x, y,
        x_label='Date',
        y_label='Delay Time δt (samples)',
        ax=ax,
        x_width=x_width,
        x_start=x_start,
        x_end=x_end,
        y_start=y_start,
        y_end=y_end,
        x_over=x_overlap,
        y_over=y_overlap,
        y_width=y_width,
        flag_y_norm=True,  # Normalize by column (Baillard's norm_y)
        flag_resample=flag_smooth,  # Enable resampling for smoothing
        flag_filter=flag_smooth,  # Enable Gaussian filtering after resampling
        x_filter_per=x_filter_per,  # Smoothing percentage in time
        y_filter_per=y_filter_per,  # Smoothing percentage in y
        filter_mode='nearest',  # Non-cyclic edge handling
        vmax=None
    )
    
    # Add eruption line
    eruption_time = UTCDateTime(2015, 4, 24, 6)
    ax.axvline(eruption_time.matplotlib_date, color='white', 
               linestyle='--', alpha=0.7, linewidth=2)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label('Normalized Density', fontsize=10)
    
    # Add title with statistics
    dt_mean = df['dt'].mean()
    dt_std = df['dt'].std()
    title = (f"Delay Time Time Series - Station {station}\n"
             f"N = {len(df)} events | "
             f"δt: {dt_mean:.3f} ± {dt_std:.3f} s "
             f"({dt_mean*sampling_rate:.1f} ± {dt_std*sampling_rate:.1f} samples)")
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    return fig, ax


def plot_phi_timeseries_movehisto(results_df, time_column='event_datetime',
                                   x_width=5*24*3600, x_overlap=0.95,
                                   y_width=9, y_overlap=0.9,
                                   figsize=(14, 6),
                                   station='AXAS2',
                                   flag_smooth=True,
                                   x_filter_per=10,
                                   y_filter_per=10):
    """
    Plot fast direction (phi) over time using moving window 2D histogram with Baillard-style smoothing.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results
    time_column : str
        Name of time column (default 'event_datetime')
    x_width : float
        Width of time window in seconds (default 5 days = 5*24*3600)
    x_overlap : float
        Overlap fraction for time windows (0.95 = 95% overlap)
    y_width : float
        Bin width for phi in degrees (default 9 = 180/20)
    y_overlap : float
        Overlap for y-direction
    figsize : tuple
        Figure size
    station : str
        Station name for title
    flag_smooth : bool
        Apply Baillard-style smoothing via resampling (default True)
    x_filter_per : float
        Gaussian filter width percentage in time direction (default 10)
    y_filter_per : float
        Gaussian filter width percentage in y direction (default 10)
    """
    import sws_methods as swm
    
    # Extract data
    df = results_df.copy()
    times = pd.to_datetime(df[time_column])
    phi_deg = df['phi'].values
    
    # Convert to UTCDateTime for movehisto2d
    x = np.array([UTCDateTime(t) for t in times])
    y = phi_deg
    
    # Set time range
    x_start = UTCDateTime(times.min())
    x_end = UTCDateTime(times.max())
    y_start = -90
    y_end = 90
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot with Baillard-style resampling and smoothing
    (ax, im, X, Y, Z, x_bins, x_diffs) = plot_movehisto2d(
        x, y,
        x_label='Date',
        y_label='Fast Direction φ (°)',
        ax=ax,
        x_width=x_width,
        x_start=x_start,
        x_end=x_end,
        y_start=y_start,
        y_end=y_end,
        x_over=x_overlap,
        y_over=y_overlap,
        y_width=y_width,
        flag_y_norm=True,  # Normalize by column (Baillard's norm_y)
        flag_resample=flag_smooth,  # Enable resampling for smoothing
        flag_filter=flag_smooth,  # Enable Gaussian filtering after resampling
        x_filter_per=x_filter_per,  # Smoothing percentage in time
        y_filter_per=y_filter_per,  # Smoothing percentage in y
        #filter_mode=['nearest', 'wrap'],  # 'wrap' for cyclic phi, 'nearest' for time
        filter_mode = ['wrap', 'nearest'],
        vmax=None
    )
    
    # Add eruption line
    eruption_time = UTCDateTime(2015, 4, 24, 6)
    ax.axvline(eruption_time.matplotlib_date, color='white', 
               linestyle='--', alpha=0.7, linewidth=2)
    
    # Add horizontal line at 0
    ax.axhline(0, color='white', linestyle='--', alpha=0.5, linewidth=1.5)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label('Normalized Density', fontsize=10)
    
    # Add title with statistics
    phi_mean = df['phi'].mean()
    phi_std = df['phi'].std()
    title = (f"Fast Direction Time Series - Station {station}\n"
             f"N = {len(df)} events | "
             f"φ: {phi_mean:.1f}° ± {phi_std:.1f}° "
             f"({np.deg2rad(phi_mean):.2f} ± {np.deg2rad(phi_std):.2f} rad)")
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    return fig, ax

In [ ]:
results_df_axec2 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_002_tmid_1_9_2_1.csv')

In [ ]:
results_df_axec2 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_002_tmid_1_9_2_1.csv')

# Convert evetn_datetime to UTCDateTime
results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: UTCDateTime(x))

# Same for AXEC2 data
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: UTCDateTime(x))

In [ ]:
results_df_axec2 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_002_tmid_1_9_2_1.csv')

# Convert evetn_datetime to UTCDateTime
results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: UTCDateTime(x))

# Same for AXEC2 data
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: UTCDateTime(x))

# Filter results_df_axec2 to be +/- 24 hrs of the eruption onset
eruption_time = UTCDateTime(2015, 4, 24, 6)
time_window = 24 * 3600  # 24 hours in seconds
start_time = eruption_time - time_window
end_time = eruption_time + time_window
mask = (results_df_axec2['event_datetime'] >= start_time) & (results_df_axec2['event_datetime'] <= end_time)
results_df_axec2 = results_df_axec2[mask]

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: x.datetime)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: x.datetime)

In [ ]:
# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

In [ ]:
# Now plot on AXEC2 data

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=180/90,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in dt (5%)
    figsize=(6, 8)
)
plt.show()

In [ ]:
# Create results_df_axec2_baillard_phi to convert conventions: go to CCW from E from clockwise from N in phi
# Create results_df_axec2_baillard_phi to convert conventions: 
# From: Clockwise from North (-90° to +90°)
# To: Counter-clockwise from East (-90° to +90°)

results_df_axec2_baillard = results_df_axec2.copy()

# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_df_axec2_baillard['phi'] = 90 - results_df_axec2['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_df_axec2_baillard['phi'] = results_df_axec2_baillard['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)

In [ ]:
# Now plot on AXEC2 data

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axec2_baillard, 
    station='AXEC2, High Resolution',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=180/90,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in dt (5%)
    figsize=(6, 8)
)
plt.show()

In [ ]:
# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axec2_baillard, 
    station='AXEC2, High Resolution',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in dt (5%)
    figsize=(6, 8)
)
plt.show()

In [ ]:
results_df = pd.read_csv('../results/splitting_results_axas2_apr_20_28_sigma_2_1_tmid_1_8_2_2.csv')

In [ ]:
# Now plot on AXAS2 data

fig, ax = plot_phi_timeseries_movehisto(
    results_df, 
    station='AXAS2, High Resolution',
    x_width=3600,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=180/90,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in dt (5%)
    figsize=(14, 6)
)
plt.show()

In [ ]:
# Convert evetn_datetime to UTCDateTime
results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: UTCDateTime(x))

# Filter results_df_axec2 to be +/- 24 hrs of the eruption onset
eruption_time = UTCDateTime(2015, 4, 24, 6)
time_window = 24 * 3600  # 24 hours in seconds
start_time = eruption_time - time_window
end_time = eruption_time + time_window
mask = (results_df['event_datetime'] >= start_time) & (results_df['event_datetime'] <= end_time)
results_df = results_df[mask]

In [ ]:
# Convert results_df event_datetime to datetime for plotting
results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: x.datetime)

In [ ]:
# Convert results_df event_datetime to datetime for plotting

results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: UTCDateTime(x))

results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: x.datetime)

# Create results_df_axas2_baillard_phi to convert conventions: go to CCW from E from clockwise from N in phi
# Create results_df_axas2_baillard_phi to convert conventions: 
# From: Clockwise from North (-90° to +90°)
# To: Counter-clockwise from East (-90° to +90°)

results_df_axas2_baillard_phi = results_df.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_df_axas2_baillard_phi['phi'] = 90 - results_df_axas2_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_df_axas2_baillard_phi['phi'] = results_df_axas2_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)

In [ ]:
# Now plot on AXAS2 data

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axas2_baillard_phi, 
    station='AXAS2, High Resolution',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=180/90,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in dt (5%)
    figsize=(6, 8)
)
plt.show()

In [ ]:
# Now plot on AXAS2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axas2_baillard_phi, 
    station='AXAS2, High Resolution',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=2,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in dt (5%)
    figsize=(6, 8)
)
plt.show()

In [ ]:
## Look at results using different parameters

In [ ]:
results_df_axec2 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_002_tmid_1_5_2_5.csv')

# Same for AXEC2 data
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: x.datetime)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution; 1.5-2.5 Tmid End',
    x_width=3600,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution; 1.5-2.5 Tmid End',
    x_width=3600,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
results_df_axec2 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_002_tmid_1_9_2_1.csv')

# Same for AXEC2 data
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: x.datetime)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution; 1.9-2.1 Tmid End',
    x_width=3600,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution; 1.9-2.1 Tmid End',
    x_width=3600,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
results_df_axec2 = pd.read_csv('../results/splitting_results_axec2_apr_20_28.csv')

# Same for AXEC2 data
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2['event_datetime'] = results_df_axec2['event_datetime'].apply(lambda x: x.datetime)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution; 2σ to S-pick time, 1.5-2.5 Tmid End',
    x_width=3600,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axec2, 
    station='AXEC2, High Resolution; 2σ to S-pick time, 1.5-2.5 Tmid End',
    x_width=3600,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
results_df = pd.read_csv('../results/splitting_results_axas2_apr_20_28_sigma_2_1_tmid_1_8_2_2.csv')

# Convert results_df event_datetime to datetime for plotting

results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: UTCDateTime(x))

results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: x.datetime)

# Create results_df_axas2_baillard_phi to convert conventions: go to CCW from E from clockwise from N in phi
# Create results_df_axas2_baillard_phi to convert conventions: 
# From: Clockwise from North (-90° to +90°)
# To: Counter-clockwise from East (-90° to +90°)

results_df_axas2_baillard_phi = results_df.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_df_axas2_baillard_phi['phi'] = 90 - results_df_axas2_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_df_axas2_baillard_phi['phi'] = results_df_axas2_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)

In [ ]:
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df,
    title_prefix=f"Fast Direction Distribution, AXAS2, SWSPy: 2-1σ, 1.8-2.2 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

In [ ]:
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_axec2_1_0_1_7_2_3,
    title_prefix=f"Fast Direction Distribution, AXEC2, SWSPy: 1-0σ, 1.7-2.3 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

In [ ]:

results_df_axec2_1_0_1_7_2_3 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_1_sigma_0_tmid_1_7_2_3.csv')

# Same for AXEC2 data
results_df_axec2_1_0_1_7_2_3['event_datetime'] = results_df_axec2_1_0_1_7_2_3['event_datetime'].apply(lambda x: UTCDateTime(x))

# Filter results_df_axec2 to be +/- 24 hrs of the eruption onset
eruption_time = UTCDateTime(2015, 4, 24, 6)
time_window = 24 * 3600  # 24 hours in seconds
start_time = eruption_time - time_window
end_time = eruption_time + time_window
mask = (results_df_axec2_1_0_1_7_2_3['event_datetime'] >= start_time) & (results_df_axec2_1_0_1_7_2_3['event_datetime'] <= end_time)
results_df_axec2_1_0_1_7_2_3 = results_df_axec2_1_0_1_7_2_3[mask]

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2_1_0_1_7_2_3['event_datetime'] = results_df_axec2_1_0_1_7_2_3['event_datetime'].apply(lambda x: x.datetime)

results_df_axec2_baillard_phi = results_df_axec2_1_0_1_7_2_3.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_df_axec2_baillard_phi['phi'] = 90 - results_df_axec2_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_df_axec2_baillard_phi['phi'] = results_df_axec2_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)

# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axec2_baillard_phi, 
    station='AXEC2, High Resolution; 1σ to S-pick time, 1.7-2.3 Tmid End',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in dt (5%)
    figsize=(6,8)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axec2_baillard_phi, 
    station='AXEC2, High Resolution; 1σ to S-pick time, 1.7-2.3 Tmid End',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=180/40,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in phi (5%)
    figsize=(6,8)
)
plt.show()


In [ ]:
results_df_axec2_2_2_1_8_2_2 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_2_sigma_1_tmid_1_8_2_2.csv')

results_df_axec2_2_2_1_8_2_2['event_datetime'] = results_df_axec2_2_2_1_8_2_2['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2_2_2_1_8_2_2['event_datetime'] = results_df_axec2_2_2_1_8_2_2['event_datetime'].apply(lambda x: x.datetime)

In [ ]:
fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_axec2_2_2_1_8_2_2,
    title_prefix=f"Fast Direction Distribution, AXEC2, SWSPy: 2-1σ, 1.8-2.2 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

In [ ]:

results_df_axec2_1_0_1_7_2_3 = pd.read_csv('../results/splitting_results_axec2_apr_20_28_2_sigma_1_tmid_1_8_2_2.csv')

# Same for AXEC2 data
results_df_axec2_1_0_1_7_2_3['event_datetime'] = results_df_axec2_1_0_1_7_2_3['event_datetime'].apply(lambda x: UTCDateTime(x))

# Filter results_df_axec2 to be +/- 24 hrs of the eruption onset
eruption_time = UTCDateTime(2015, 4, 24, 6)
time_window = 24 * 3600  # 24 hours in seconds
start_time = eruption_time - time_window
end_time = eruption_time + time_window
mask = (results_df_axec2_1_0_1_7_2_3['event_datetime'] >= start_time) & (results_df_axec2_1_0_1_7_2_3['event_datetime'] <= end_time)
results_df_axec2_1_0_1_7_2_3 = results_df_axec2_1_0_1_7_2_3[mask]

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2_1_0_1_7_2_3['event_datetime'] = results_df_axec2_1_0_1_7_2_3['event_datetime'].apply(lambda x: x.datetime)

results_df_axec2_baillard_phi = results_df_axec2_1_0_1_7_2_3.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_df_axec2_baillard_phi['phi'] = 90 - results_df_axec2_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_df_axec2_baillard_phi['phi'] = results_df_axec2_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)

# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axec2_baillard_phi, 
    station='AXEC2, High Resolution; 2-1σ, 1.8-2.2 Tmid End',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in dt (5%)
    figsize=(6,8)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axec2_baillard_phi, 
    station='AXEC2, High Resolution; 2-1σ, 1.8-2.2 Tmid End',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=180/40,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in phi (5%)
    figsize=(6,8)
)
plt.show()


In [ ]:
results_df_axec2_2_0_1_5_2_5 = pd.read_csv('../results/splitting_results_axec2_apr_20_28.csv')

results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: x.datetime)

fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_axec2_2_0_1_5_2_5,
    title_prefix=f"Fast Direction Distribution, AXEC2, SWSPy: 2-0σ, 1.5-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

In [ ]:
results_df_axec2_2_0_1_5_2_5 = pd.read_csv('../results/splitting_results_axec2_apr_20_28.csv')

results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: x.datetime)

fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
    results_df_axec2_2_0_1_5_2_5,
    title_prefix=f"Fast Direction Distribution, AXEC2, SWSPy: 2-0σ, 1.5-2.5 Tmid End",
    nbins=36,  # 10° bins
    color='steelblue',
    figsize=(16, 7)
)
plt.show()

results_df_axec2_2_0_1_5_2_5 = pd.read_csv('../results/splitting_results_axec2_apr_20_28.csv')

results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: UTCDateTime(x))

# Filter results_df_axec2 to be +/- 24 hrs of the eruption onset
eruption_time = UTCDateTime(2015, 4, 24, 6)
time_window = 24 * 3600  # 24 hours in seconds
start_time = eruption_time - time_window
end_time = eruption_time + time_window
mask = (results_df_axec2_2_0_1_5_2_5['event_datetime'] >= start_time) & (results_df_axec2_2_0_1_5_2_5['event_datetime'] <= end_time)
results_df_axec2_2_0_1_5_2_5 = results_df_axec2_2_0_1_5_2_5[mask]

# Convert results_df_axec2 event_datetime to datetime for plotting
results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: x.datetime)

results_df_axec2_baillard_phi = results_df_axec2_2_0_1_5_2_5.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_df_axec2_baillard_phi['phi'] = 90 - results_df_axec2_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_df_axec2_baillard_phi['phi'] = results_df_axec2_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)

# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_df_axec2_baillard_phi, 
    station='AXEC2, High Resolution; 2-0σ, 1.5-2.5 Tmid End',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in dt (5%)
    figsize=(6,8)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_df_axec2_baillard_phi, 
    station='AXEC2, High Resolution; 2-0σ, 1.5-2.5 Tmid End',
    x_width=100,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=180/40,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5,      # Less smoothing in phi (5%)
    figsize=(6,8)
)
plt.show()


In [ ]:
## Do it for every AXAS2 file
def output_plots(filename, first_start, last_start, first_end, last_end, station):
    results_df_axec2_2_0_1_5_2_5 = pd.read_csv(filename)

    results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: UTCDateTime(x))

    # Convert results_df_axec2 event_datetime to datetime for plotting
    results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: x.datetime)

    fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
        results_df_axec2_2_0_1_5_2_5,
        title_prefix="Fast Direction Distribution, " + str(station) + ", SWSPy: " + str(first_start) + "-"+ str(last_start) + "σ, " + str(first_end) + "-" + str(last_end) + " Tmid End",
        nbins=36,  # 10° bins
        color='steelblue',
        figsize=(16, 7)
    )
    plt.show()

    results_df_axec2_2_0_1_5_2_5 = pd.read_csv(filename)

    results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: UTCDateTime(x))

    # Filter results_df_axec2 to be +/- 24 hrs of the eruption onset
    eruption_time = UTCDateTime(2015, 4, 24, 6)
    time_window = 24 * 3600  # 24 hours in seconds
    start_time = eruption_time - time_window
    end_time = eruption_time + time_window
    mask = (results_df_axec2_2_0_1_5_2_5['event_datetime'] >= start_time) & (results_df_axec2_2_0_1_5_2_5['event_datetime'] <= end_time)
    results_df_axec2_2_0_1_5_2_5 = results_df_axec2_2_0_1_5_2_5[mask]

    # Convert results_df_axec2 event_datetime to datetime for plotting
    results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: x.datetime)

    results_df_axec2_baillard_phi = results_df_axec2_2_0_1_5_2_5.copy()
    # Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
    results_df_axec2_baillard_phi['phi'] = 90 - results_df_axec2_baillard_phi['phi']

    # Handle wrapping: keep values in -90° to +90° range
    # If result > 90°, subtract 180°
    # If result < -90°, add 180°
    results_df_axec2_baillard_phi['phi'] = results_df_axec2_baillard_phi['phi'].apply(
        lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
    )

    # Now plot on AXEC2 data

    fig, ax = plot_dt_timeseries_movehisto(
        results_df_axec2_baillard_phi, 
        station=str(station) + ", SWSPy: " + str(first_start) + "-"+ str(last_start) + "σ, " + str(first_end) + "-" + str(last_end) + " Tmid End",
        x_width=600,  # 1 day time window (shorter)
        x_overlap=0.9,
        y_width=1,          # 1 sample bins (finer)
        y_overlap=0.9,
        flag_smooth=True,
        x_filter_per=5,     # Less smoothing in time (5%)
        y_filter_per=5,      # Less smoothing in dt (5%)
        figsize=(6,8)
    )
    plt.show()

    fig, ax = plot_phi_timeseries_movehisto(
        results_df_axec2_baillard_phi, 
        station=str(station) + ", SWSPy: " + str(first_start) + "-"+ str(last_start) + "σ, " + str(first_end) + "-" + str(last_end) + " Tmid End",
        x_width=600,  # 1 day time window (shorter)
        x_overlap=0.9,
        y_width=180/40,          # 5 degree bins (finer)
        y_overlap=0.9,
        flag_smooth=True,
        x_filter_per=5,     # Less smoothing in time (5%)
        y_filter_per=5,      # Less smoothing in phi (5%)
        figsize=(6,8)
    )
    plt.show()

    return results_df_axec2_2_0_1_5_2_5


In [ ]:

output_plots(filename='../results/splitting_results_axec2_apr_20_28_1_sigma_0_tmid_1_7_2_3.csv',
             first_start=1, last_start=0, first_end=1.7, last_end=2.3, station='AXEC2')

In [ ]:

output_plots(filename='../results/splitting_results_axec2_apr_20_28_2_sigma_1_tmid_1_8_2_2.csv',
             first_start=2, last_start=1, first_end=1.8, last_end=2.2)

In [ ]:
output_plots(filename='../results/splitting_results_axas2_apr_20_28_sigma_2_validation.csv',
             first_start=2, last_start=0, first_end=1.5, last_end=2.5)

In [ ]:
output_plots(filename='../results/splitting_results_axas2_apr_20_28_3_sigma_start_1_sigma_end.csv',
             first_start=3, last_start=1, first_end=1.5, last_end=2.5)

In [ ]:
output_plots(filename='../results/splitting_results_axas2_apr_20_28_3_sigma_start.csv',
             first_start=3, last_start=0, first_end=1.5, last_end=2.5)

In [ ]:
output_plots(filename='../results/splitting_results_axas2_apr_20_28_1_tmid_start.csv',
             first_start=2, last_start=0, first_end=1.0, last_end=2.5)

In [ ]:
'splitting_results_axec2_apr_20_28_2_sigma_1_tmid_1_8_2_2


In [ ]:
output_plots(filename='../results/splitting_results_axas2_apr_20_28_baillard.csv', first_start='0.02s', last_start='0', first_end='2', last_end='2')

In [ ]:
output_plots(filename='../results/splitting_results_axec2_apr_20_28_2_sigma_1_tmid_1_6_2_4.csv',
             first_start = 2, last_start= 1, first_end=1.6, last_end=2.4, station='AXEC2')

In [ ]:
results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd.csv')

In [ ]:
display(results_mldd)

In [ ]:
results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

In [ ]:
display(results_mldd)

In [ ]:
results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

In [ ]:
display(results_mldd)

In [ ]:
results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd_cleaned.csv', index=False)

In [ ]:
results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec2_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXEC2 MLDD')

In [ ]:
output_plots(filename='../results/splitting_results_baillard_axec2_apr_20_28_mldd.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXEC2 MLDD Baillard')

In [ ]:

# Same for AXEC2 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
results_baillard_mldd = pd.read_csv('../results/splitting_results_baillard_axec2_apr_20_28_mldd.csv')

In [ ]:

# Same for AXEC2 data
results_baillard_mldd['event_datetime'] = results_baillard_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_baillard_mldd['event_datetime'] = results_baillard_mldd['event_datetime'].apply(lambda x: x.datetime)

# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_baillard_mldd, 
    station='AXEC2, MLDD Catalog; Baillard Method',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_baillard_mldd, 
    station='AXEC2, MLDD Catalog; Baillard Method',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
results_mldd_axec3 = pd.read_csv('../results/splitting_results_swspy_axec3_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd_axec3 = results_mldd_axec3.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd_axec3.columns if 'window' in col], axis=1)

results_mldd_axec3 = results_mldd_axec3.dropna()

results_mldd_axec3.to_csv('../results/splitting_results_swspy_axec3_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec3_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXEC3 MLDD')

In [ ]:

# Same for AXEC3 data
results_mldd_axec3['event_datetime'] = results_mldd_axec3['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec3 event_datetime to datetime for plotting
results_mldd_axec3['event_datetime'] = results_mldd_axec3['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd_axec3.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXEC3 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC3, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC3, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
results_mldd_axec3_baillard = pd.read_csv('../results/splitting_results_baillard_axec3_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name

results_mldd_axec3_baillard = results_mldd_axec3_baillard.dropna()

results_mldd_axec3_baillard.to_csv('../results/splitting_results_baillard_axec3_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_baillard_axec3_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXEC3 MLDD, Baillard Method')

In [ ]:

# Same for AXEC3 data
results_mldd_axec3_baillard['event_datetime'] = results_mldd_axec3_baillard['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec3 event_datetime to datetime for plotting
results_mldd_axec3_baillard['event_datetime'] = results_mldd_axec3_baillard['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd_axec3_baillard.copy()


# Now plot on AXEC3 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC3, MLDD Catalog; Baillard Method',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC3, MLDD Catalog; Baillard Method',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
## Look at a wider range of end windowing results for AXEC2 with MLDD catalog in the week around the eruption.

results_mldd = pd.read_csv('../results/splitting_results_swspy_1_5_2_5_axec2_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_1_5_2_5_axec2_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_1_5_2_5_axec2_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.5, last_end=2.5, station='AXEC2 MLDD')

In [ ]:

# Same for AXEC2 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.5-2.5 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.5-2.5 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
## Look at a wider range of end windowing results for AXEC3 with MLDD catalog in the week around the eruption.

results_mldd = pd.read_csv('../results/splitting_results_swspy_axec3_apr_20_28_mldd_1_5_2_5.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_1_5_2_5_axec3_apr_20_28_mldd_1_5_2_5_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_1_5_2_5_axec3_apr_20_28_mldd_1_5_2_5_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.5, last_end=2.5, station='AXEC3 MLDD')

In [ ]:

# Same for AXEC2 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC3, SWSPy, MLDD Catalog; 2-1σ, 1.5-2.5 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=2,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC3, SWSPy, MLDD Catalog; 2-1σ, 1.5-2.5 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
## Look at a wider range of end windowing results for AXAS1 with MLDD catalog in the week around the eruption.

results_mldd = pd.read_csv('../results/splitting_results_swspy_axas1_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_1_5_2_5_axas1_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_1_5_2_5_axas1_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXAS1 MLDD')

In [ ]:

# Same for AXAS1 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS1, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS1, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:


## Look at a wider range of end windowing results for AXAS1 with MLDD catalog in the week around the eruption.

results_mldd = pd.read_csv('../results/splitting_results_swspy_axcc1_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axcc1_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axcc1_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXCC1 MLDD')

In [ ]:

# Same for AXCC1 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXCC1, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXCC1, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:

## Look at a wider range of end windowing results for AXAS1 with MLDD catalog in the week around the eruption.

results_mldd = pd.read_csv('../splitting_results_swspy_axec2_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec2_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec2_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXEC2 MLDD')

In [ ]:

# Same for AXEC2 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXEC2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600 * 24 * 5,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600 * 24 * 5,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
###############

In [ ]:
# Look at AXEC1 data with MLDD catalog in the week around the eruption.

results_mldd = pd.read_csv('../results/splitting_results_swspy_axec1_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec1_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec1_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXEC1 MLDD')

In [ ]:

# Same for AXEC1 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axec1 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXEC1 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC1, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC1, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
# Look at AXID1 data with MLDD catalog in the week around the eruption.

results_mldd = pd.read_csv('../results/splitting_results_swspy_axid1_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axid1_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axid1_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXID1 MLDD')

In [ ]:

# Same for AXID1 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axid1 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXID1 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXID1, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXID1, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
# Look at AXAS2 data with MLDD catalog in the week around the eruption.

results_mldd = pd.read_csv('../results/splitting_results_swspy_axas2_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axas2_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axas2_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXAS2 MLDD')

In [ ]:

# Same for AXAS2 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXAS2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
# Look at AXAS2 data with MLDD catalog in the week around the eruption.

results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd_15_25_eps_5_samp_10.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd_15_25_eps_5_samp_10_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec2_apr_20_28_mldd_15_25_eps_5_samp_10_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.5, last_end=2.5, station='AXEC2 MLDD, Eps=5%, Min Samp=10')

In [ ]:

# Same for AXAS2 data
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


# Now plot on AXAS2 data

fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS2, SWSPy, MLDD Catalog; Eps=5%, Min Samp=10, 1.5-2.5 Tmid End',
    x_width=3600/4,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS2, SWSPy, MLDD Catalog; Eps=5%, Min Samp=10, 1.5-2.5 Tmid End',
    x_width=3600/4,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

## Now we will plot the five reliable stations for 1.9-2.1 Tmid, to compare the 24hr plots vs. Christian's results


In [ ]:
# AXAS1

results_mldd = pd.read_csv('../results/splitting_results_swspy_axas1_apr_20_28_mldd_19_21.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axas1_apr_20_28_mldd_19_21_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axas1_apr_20_28_mldd_19_21_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.9, last_end=2.1, station='AXAS1 MLDD')

results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS1, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS1, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
# AXAS2

results_mldd = pd.read_csv('../results/splitting_results_swspy_axas2_apr_20_28_mldd_19_21.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axas2_apr_20_28_mldd_19_21_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axas2_apr_20_28_mldd_19_21_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.9, last_end=2.1, station='AXAS2 MLDD')

results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS2, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXAS2, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
# AXEC1

results_mldd = pd.read_csv('../results/splitting_results_swspy_axec1_apr_20_28_mldd_19_21.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec1_apr_20_28_mldd_19_21_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec1_apr_20_28_mldd_19_21_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.9, last_end=2.1, station='AXEC1 MLDD')

results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC1, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC1, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
# AXEC2

results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd_19_21.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd_19_21_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec2_apr_20_28_mldd_19_21_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.9, last_end=2.1, station='AXEC2 MLDD')

results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
# AXEC3

results_mldd = pd.read_csv('../results/splitting_results_swspy_axec3_apr_20_28_mldd_19_21.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec3_apr_20_28_mldd_19_21_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec3_apr_20_28_mldd_19_21_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.9, last_end=2.1, station='AXEC3 MLDD')

results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC3, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC3, SWSPy, MLDD Catalog; 1.9-2.1 Tmid End',
    x_width=3600/2,  # 1 day time window (shorter)
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

## Now we will look at our longest plot yet, showing AXEC2 from Jan 22 - Jun 01, 2015

In [ ]:
# AXEC2 - Across the whole eruption!

results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_mldd_full_eruption.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec2_mldd_full_eruption_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec2_mldd_full_eruption_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXEC2 MLDD')

results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 1.8-2.2 Tmid End',
    x_width=3600 * 24 * 3,  # 3 day time window
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 1.8-2.2 Tmid End',
    x_width=3600 * 24 * 3,  # 3 day time window
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=5      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:
def plot_movehisto2d(x,y,
                x_label='X',y_label='Y',title='',ax=None,vmax=None,
                **movehisto2d_kwargs):
    """
    Function made to plot an histo2d but using a moving window in both directions, this ensure better
    consistency between neighbor bins. 
    The histogram can also work when data is an obspy.UTCDateTime array, then the x_start and x_end must
    be given in UTCDateTime as well and the x_width should be given in seconds.
    UTCDatetime are converted to timestamps (seconds since 1970)
    
    Inputs
    ------
        x,y: np.array: arrays containing the data to apply histogram on (x can be UTCDateTime)
        x_width,y_width: float: width of the bins (in seconds for UTCDateTime)
        [x,y]_[start,end]: float: start and end for histogram edges
        [x,y]_over: float in [0,1]: overlap for windows [1 = full overlap]
        flag_y_norm: Boolean: True to normalize by the maximum in each column
        flag_resample: Boolean: Enable resampling to uniform grid (required for smoothing)
        flag_filter: Boolean: Enable Gaussian filtering (requires flag_resample=True)
        x_filter_per, y_filter_per: float in [0,100]: width percentage for smoothing
        filter_mode: str or list: 'nearest' or 'wrap' for edge handling
        [x,y]_label: str
        
    Ouputs
    ------
        ax, im, X, Y, Z, x_bins, x_diffs
        
    Comments:
    ---------
        Uses Baillard's movehisto2d_bin for resampling and filtering
        
    UsedIn
    ------
        SWSCat.plot_movehisto2d_time, wrapper functions
    """
    
    ### Check if is made of UTCDateTimes
    
    time_flag=False
    if isinstance(x[0],UTCDateTime):
        time_flag=True
        print('X is in UTCDateTime')
        if movehisto2d_kwargs.get('x_mode','window')=='window':
            print('Remember width should be given in seconds, otherwise memory error')
    
    ### Modify x_start and x_end, and x if x is time and convert to timestamps
    x_start=movehisto2d_kwargs.get('x_start',None)
    x_end=movehisto2d_kwargs.get('x_end',None)
    y_start=movehisto2d_kwargs.get('y_start',None)
    y_end=movehisto2d_kwargs.get('y_end',None)
    
    if time_flag:
        if (x_start is not None) & (not isinstance(x_start,UTCDateTime)):
            raise ValueError('x_start must be given in obspy.UTCDateTime')
        if (x_end is not None) & (not isinstance(x_end,UTCDateTime)):
            raise ValueError('x_end must be given in obspy.UTCDateTime')
        x=np.array([value.timestamp for value in x]) # (seconds since 1970-01-01T00:00:00)
        x_start=x_start.timestamp if x_start is not None else None
        x_end=x_end.timestamp if x_end is not None else None
        movehisto2d_kwargs['x_start']=x_start
        movehisto2d_kwargs['x_end']=x_end

    ### Bin the data (smoothing handled inside movehisto2d_bin via resampling + filtering)
    
    (X,Y,Z,x_bins,x_diffs)=swm.movehisto2d_bin(x,y,**movehisto2d_kwargs)

    # Added to fix edge artifacts issues - bins on edges have less data, highly sensitive to outliers
    # Drop first and last x-bins (edge time windows) before plotting
    if X.shape[1] > 2:          # only if we have at least 3 columns
        X = X[:, 1:-1]
        Y = Y[:, 1:-1]
        Z = Z[:, 1:-1]
        x_bins = x_bins[1:-1]
        x_diffs = x_diffs[1:-1]
    
    ########################
    #### Start plotting ####

    #### Grid spec
    
    bottom=0.15 if time_flag else 0.1
    
    ### Checks
    
    if ax is None:
        fig,ax = plt.subplots(gridspec_kw={'bottom':bottom,'left':0.15})
        
    if time_flag:
        X=np.array(swm.timestamp2matplotlib(X.ravel())).reshape(X.shape) # transform for plotting
        x_bins=swm.timestamp2matplotlib(x_bins)
        plt.setp( ax.xaxis.get_majorticklabels(), rotation=30 ,ha='right')
        ax.set_xlim(swm.timestamp2matplotlib([x_start,x_end]))
    else:
        ax.set_xlim([x_start,x_end])
       
    (Xm,Ym)=swm.XY2XY_pcolormesh(X,Y) # To ensure Pcolormesh will be centered on bins

    #im=ax.pcolormesh(X,Y,Z,cmap=plt.cm.get_cmap('jet'),rasterized=True)
    im=ax.pcolormesh(Xm,Ym,Z,cmap=plt.cm.get_cmap('inferno'),rasterized=True,vmax=vmax)
    
    ax.set_ylim([y_start,y_end])
    
    ax.set_aspect('auto')
    if time_flag:
        ax.xaxis_date()
            
    ### Cosmetic
    
    ax.set_ylabel(y_label) 
    if not time_flag:
        ax.set_xlabel(x_label) 
   
    ###### Return
    
    return (ax,im,X,Y,Z,x_bins,x_diffs)

    
def plot_fast_direction_rose_eruption_comparison(results_df, eruption_time=None, 
                                                  title_prefix="Fast Direction Distribution",
                                                  nbins=36, figsize=(16, 7), color='steelblue',
                                                  edgecolor='black', linewidth=0.5,
                                                  time_column='event_datetime'):
    """
    Create side-by-side 360° rose plots comparing fast directions before and after eruption.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results containing 'phi' and time columns
    eruption_time : UTCDateTime or str
        Time of eruption onset (default: 2015-04-24T06:00:00)
    title_prefix : str
        Prefix for plot titles
    nbins : int
        Number of angular bins (default 36 = 10° bins for 360°)
    figsize : tuple
        Figure size (width, height) for combined plot
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    time_column : str
        Name of the time column in results_df (default 'event_datetime')
    """
    if eruption_time is None:
        eruption_time = UTCDateTime(2015, 4, 24, 6)
    elif not isinstance(eruption_time, UTCDateTime):
        eruption_time = UTCDateTime(eruption_time)
    
    # Convert time column to UTCDateTime for comparison
    df_time = results_df[time_column].apply(lambda x: UTCDateTime(x) if not isinstance(x, UTCDateTime) else x)
    results_before = results_df[df_time < eruption_time]
    results_after = results_df[df_time >= eruption_time]
    
    # Create figure with two subplots
    fig = plt.figure(figsize=figsize)
    
    # Before eruption plot (left)
    ax1 = fig.add_subplot(121, projection='polar')
    plot_rose_subplot(results_before, ax1, f"{title_prefix}\nBefore Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    # After eruption plot (right)
    ax2 = fig.add_subplot(122, projection='polar')
    plot_rose_subplot(results_after, ax2, f"{title_prefix}\nAfter Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    plt.tight_layout()
    return fig, (ax1, ax2), (results_before, results_after)


def plot_rose_subplot(results_df, ax, title, nbins, color, edgecolor, linewidth):
    """
    Helper function to plot rose diagram on a given axis.
    """
    if len(results_df) == 0:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, 
                ha='center', va='center', fontsize=14)
        return
    
    fast_directions = []
    for phi in results_df['phi'].values:  # in degrees (-90 to +90)
        # Convert to 0-360 range and add 180° symmetry
        phi_0_180 = phi + 90  # Convert to 0-180 range
        phi_rad_1 = np.deg2rad(phi_0_180)
        phi_rad_2 = np.deg2rad(phi_0_180 + 180)  # Add 180° symmetric value
        fast_directions.append(phi_rad_1)
        fast_directions.append(phi_rad_2)
    
    fast_directions = np.array(fast_directions)
    
    # Create histogram bins (0 to 2π for 0° to 360°)
    bins = np.linspace(0, 2*np.pi, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = 2 * np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(-1)
    
    # Set radial ticks
    ax.set_rlabel_position(45)
    
    # Add degree labels for full 360°
    tick_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
    tick_positions = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Adjust y-limit
    max_count = counts.max()
    ax.set_ylim(0, max_count * 1.2)
    
    # Add title with statistics
    n_measurements = len(fast_directions) // 2  # Divide by 2 since we doubled for symmetry
    # Calculate circular mean
    original_directions = np.deg2rad(results_df['phi'].values)
    mean_direction = np.rad2deg(np.arctan2(np.sin(original_directions).sum(), 
                                           np.cos(original_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)

def plot_dt_timeseries_movehisto(results_df, time_column='event_datetime',
                                  x_width=5*24*3600, x_overlap=0.95,
                                  y_width=2, y_overlap=0.9,
                                  sampling_rate=200.0,
                                  figsize=(14, 6),
                                  station='AXAS2',
                                  flag_smooth=True,
                                  x_filter_per=10,
                                  y_filter_per=10):
    """
    Plot delay time (dt) over time using moving window 2D histogram with Baillard-style smoothing.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results
    time_column : str
        Name of time column (default 'event_datetime')
    x_width : float
        Width of time window in seconds (default 5 days = 5*24*3600)
    x_overlap : float
        Overlap fraction for time windows (0.95 = 95% overlap)
    y_width : float
        Bin width for dt in samples
    y_overlap : float
        Overlap for y-direction
    sampling_rate : float
        Sampling rate in Hz (default 200 Hz)
    figsize : tuple
        Figure size
    station : str
        Station name for title
    flag_smooth : bool
        Apply Baillard-style smoothing via resampling (default True)
    x_filter_per : float
        Gaussian filter width percentage in time direction (default 10)
    y_filter_per : float
        Gaussian filter width percentage in y direction (default 10)
    """
    import sws_methods as swm
    # Extract data
    df = results_df.copy()
    times = pd.to_datetime(df[time_column])
    dt_samples = df['dt'].values * sampling_rate
    
    # Convert to UTCDateTime for movehisto2d
    x = np.array([UTCDateTime(t) for t in times])
    y = dt_samples
    
    # Set time range
    x_start = UTCDateTime(times.min())
    x_end = UTCDateTime(times.max())
    y_start = 0
    y_end = 30

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot with Baillard-style resampling and smoothing
    (ax, im, X, Y, Z, x_bins, x_diffs) = plot_movehisto2d(
        x, y,
        x_label='Date',
        y_label='Delay Time δt (samples)',
        ax=ax,
        x_width=x_width,
        x_start=x_start,
        x_end=x_end,
        y_start=y_start,
        y_end=y_end,
        x_over=x_overlap,
        y_over=y_overlap,
        y_width=y_width,
        flag_y_norm=True,  # Normalize by column (Baillard's norm_y)
        flag_resample=flag_smooth,  # Enable resampling for smoothing
        flag_filter=flag_smooth,  # Enable Gaussian filtering after resampling
        x_filter_per=x_filter_per,  # Smoothing percentage in time
        y_filter_per=y_filter_per,  # Smoothing percentage in y
        filter_mode='nearest',  # Non-cyclic edge handling
        vmax=None
    )
    
    # Add eruption line
    eruption_time = UTCDateTime(2015, 4, 24, 6)
    ax.axvline(eruption_time.matplotlib_date, color='white', 
               linestyle='--', alpha=0.7, linewidth=2)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label('Normalized Density', fontsize=10)
    
    # Add title with statistics
    dt_mean = df['dt'].mean()
    dt_std = df['dt'].std()
    title = (f"Delay Time Time Series - Station {station}\n"
             f"N = {len(df)} events | "
             f"δt: {dt_mean:.3f} ± {dt_std:.3f} s "
             f"({dt_mean*sampling_rate:.1f} ± {dt_std*sampling_rate:.1f} samples)")
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    return fig, ax


def plot_phi_timeseries_movehisto(results_df, time_column='event_datetime',
                                   x_width=5*24*3600, x_overlap=0.95,
                                   y_width=9, y_overlap=0.9,
                                   figsize=(14, 6),
                                   station='AXAS2',
                                   flag_smooth=True,
                                   x_filter_per=10,
                                   y_filter_per=10):
    """
    Plot fast direction (phi) over time using moving window 2D histogram with Baillard-style smoothing.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame with splitting results
    time_column : str
        Name of time column (default 'event_datetime')
    x_width : float
        Width of time window in seconds (default 5 days = 5*24*3600)
    x_overlap : float
        Overlap fraction for time windows (0.95 = 95% overlap)
    y_width : float
        Bin width for phi in degrees (default 9 = 180/20)
    y_overlap : float
        Overlap for y-direction
    figsize : tuple
        Figure size
    station : str
        Station name for title
    flag_smooth : bool
        Apply Baillard-style smoothing via resampling (default True)
    x_filter_per : float
        Gaussian filter width percentage in time direction (default 10)
    y_filter_per : float
        Gaussian filter width percentage in y direction (default 10)
    """
    import sws_methods as swm
    
    # Extract data
    df = results_df.copy()
    times = pd.to_datetime(df[time_column])
    phi_deg = df['phi'].values
    
    # Convert to UTCDateTime for movehisto2d
    x = np.array([UTCDateTime(t) for t in times])
    y = phi_deg
    
    # Set time range
    x_start = UTCDateTime(times.min())
    x_end = UTCDateTime(times.max())
    y_start = -90
    y_end = 90
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot with Baillard-style resampling and smoothing
    (ax, im, X, Y, Z, x_bins, x_diffs) = plot_movehisto2d(
        x, y,
        x_label='Date',
        y_label='Fast Direction φ (°)',
        ax=ax,
        x_width=x_width,
        x_start=x_start,
        x_end=x_end,
        y_start=y_start,
        y_end=y_end,
        x_over=x_overlap,
        y_over=y_overlap,
        y_width=y_width,
        flag_y_norm=True,  # Normalize by column (Baillard's norm_y)
        flag_resample=flag_smooth,  # Enable resampling for smoothing
        flag_filter=flag_smooth,  # Enable Gaussian filtering after resampling
        x_filter_per=x_filter_per,  # Smoothing percentage in time
        y_filter_per=y_filter_per,  # Smoothing percentage in y
        #filter_mode=['nearest', 'wrap'],  # 'wrap' for cyclic phi, 'nearest' for time
        filter_mode = ['wrap', 'nearest'],
        vmax=None
    )
    
    # Add eruption line
    eruption_time = UTCDateTime(2015, 4, 24, 6)
    ax.axvline(eruption_time.matplotlib_date, color='white', 
               linestyle='--', alpha=0.7, linewidth=2)
    
    # Add horizontal line at 0
    ax.axhline(0, color='white', linestyle='--', alpha=0.5, linewidth=1.5)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label('Normalized Density', fontsize=10)
    
    # Add title with statistics
    phi_mean = df['phi'].mean()
    phi_std = df['phi'].std()
    title = (f"Fast Direction Time Series - Station {station}\n"
             f"N = {len(df)} events | "
             f"φ: {phi_mean:.1f}° ± {phi_std:.1f}° "
             f"({np.deg2rad(phi_mean):.2f} ± {np.deg2rad(phi_std):.2f} rad)")
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    return fig, ax

## Do it for every AXAS2 file
def output_plots(filename, first_start, last_start, first_end, last_end, station):
    results_df = pd.read_csv(filename)

    results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: UTCDateTime(x))

    # Convert results_df event_datetime to datetime for plotting
    results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: x.datetime)

    fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
        results_df,
        title_prefix="Fast Direction Distribution, " + str(station) + ", SWSPy: " + str(first_start) + "-"+ str(last_start) + "σ, " + str(first_end) + "-" + str(last_end) + " Tmid End",
        nbins=36,  # 10° bins
        color='steelblue',
        figsize=(16, 7)
    )
    plt.show()

    results_df = pd.read_csv(filename)

    results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: UTCDateTime(x))

    # Filter results_df to be +/- 24 hrs of the eruption onset
    eruption_time = UTCDateTime(2015, 4, 24, 6)
    time_window = 24 * 3600  # 24 hours in seconds
    start_time = eruption_time - time_window
    end_time = eruption_time + time_window
    mask = (results_df['event_datetime'] >= start_time) & (results_df['event_datetime'] <= end_time)
    results_df = results_df[mask]

    # Convert results_df event_datetime to datetime for plotting
    results_df['event_datetime'] = results_df['event_datetime'].apply(lambda x: x.datetime)

    results_df_baillard_phi = results_df.copy()
    # Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
    results_df_baillard_phi['phi'] = 90 - results_df_baillard_phi['phi']

    # Handle wrapping: keep values in -90° to +90° range
    # If result > 90°, subtract 180°
    # If result < -90°, add 180°
    results_df_baillard_phi['phi'] = results_df_baillard_phi['phi'].apply(
        lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
    )

    # Now plot on AXEC2 data

    fig, ax = plot_dt_timeseries_movehisto(
        results_df_baillard_phi, 
        station=str(station) + ", SWSPy: " + str(first_start) + "-"+ str(last_start) + "σ, " + str(first_end) + "-" + str(last_end) + " Tmid End",
        x_width=100,  # 600 seconds
        x_overlap=0.9,
        y_width=1,          # 1 sample bins (finer)
        y_overlap=0.9,
        flag_smooth=True,
        x_filter_per=5,     # Less smoothing in time (5%)
        y_filter_per=3,      # Less smoothing in dt (5%)
        figsize=(6,8)
    )
    plt.show()

    fig, ax = plot_phi_timeseries_movehisto(
        results_df_baillard_phi, 
        station=str(station) + ", SWSPy: " + str(first_start) + "-"+ str(last_start) + "σ, " + str(first_end) + "-" + str(last_end) + " Tmid End",
        x_width=100,  # 600 seconds
        x_overlap=0.9,
        y_width=180/40,          # 5 degree bins (finer)
        y_overlap=0.9,
        flag_smooth=True,
        x_filter_per=5,     # Less smoothing in time (5%)
        y_filter_per=3,      # Less smoothing in phi (5%)
        figsize=(6,8)
    )
    plt.show()

    return results_df


In [ ]:
# Now look at AXEC2, MLDD, with really tight windows like Christian
# AXEC2 - Across the whole eruption!

results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec2_apr_20_28_mldd_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXEC2 MLDD')

results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600 / 2 ,  # 3 day time window
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=4      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End',
    x_width=3600 / 2 ,  # 3 day time window
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=4      # Less smoothing in phi (5%)
)
plt.show()

In [ ]:

# Now look at AXEC2, MLDD, with optimal parameters
results_mldd = pd.read_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd_Eps_15_Samp_15.csv')
# Drop dominant_period, first_window_start_pre_S_pick, last_window_start_pre_S_pick, first_window_end_post_S_pick, last_window_end_pre_S_pick, and all columns with "window" in their name
results_mldd = results_mldd.drop(['dominant_period', 'first_window_start_pre_S_pick', 'last_window_start_pre_S_pick', 'first_window_end_post_S_pick', 'last_window_end_post_S_pick'] + [col for col in results_mldd.columns if 'window' in col], axis=1)

results_mldd = results_mldd.dropna()

results_mldd.to_csv('../results/splitting_results_swspy_axec2_apr_20_28_mldd_Eps_15_Samp_15_cleaned.csv', index=False)

output_plots(filename='../results/splitting_results_swspy_axec2_apr_20_28_mldd_Eps_15_Samp_15_cleaned.csv',
             first_start = 2, last_start= 1, first_end=1.8, last_end=2.2, station='AXEC2 MLDD')

results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: UTCDateTime(x))

# Convert results_df_axas2 event_datetime to datetime for plotting
results_mldd['event_datetime'] = results_mldd['event_datetime'].apply(lambda x: x.datetime)

results_mldd_baillard_phi = results_mldd.copy()
# Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
results_mldd_baillard_phi['phi'] = 90 - results_mldd_baillard_phi['phi']

# Handle wrapping: keep values in -90° to +90° range
# If result > 90°, subtract 180°
# If result < -90°, add 180°
results_mldd_baillard_phi['phi'] = results_mldd_baillard_phi['phi'].apply(
    lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
)


fig, ax = plot_dt_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End; Eps= 0.15, Min Samp=15',
    x_width=3600 / 2 ,  # 3 day time window
    x_overlap=0.9,
    y_width=1,          # 1 sample bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=4      # Less smoothing in dt (5%)
)
plt.show()

fig, ax = plot_phi_timeseries_movehisto(
    results_mldd_baillard_phi, 
    station='AXEC2, SWSPy, MLDD Catalog; 2-1σ, 1.8-2.2 Tmid End; Eps= 0.15, Min Samp=15',
    x_width=3600 / 2 ,  # 3 day time window
    x_overlap=0.9,
    y_width=5,          # 5 degree bins (finer)
    y_overlap=0.9,
    flag_smooth=True,
    x_filter_per=5,     # Less smoothing in time (5%)
    y_filter_per=4      # Less smoothing in phi (5%)
)
plt.show()